# Import 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
%matplotlib inline 
# pour afficher facilement les graphiques 
from urllib.request import urlopen
from bs4 import BeautifulSoup
import xml.etree.ElementTree as ET
import os




# FONCTIONS 

261 = numero du bullettin 

est ce que la description de l'image c'est juste le texte en bleu ou aussi celui en gris ?
est ce que l'auteur / rédateur c'est juste le nom premon ou alors c'est toute la ligne du rédacteur 

In [ ]:

def recuperation_date(fichier) :
    try : 
        # obtenir le code html de la page
        with open(fichier, "r", encoding = "UTF8") as f :
            html = f.read()

        # cree un objet beautifulSoup en transmettant le code html à la fonction BeautifulSoup()
        soup = BeautifulSoup(html, 'html.parser' )
        type(soup)

        # récuperer l'element dans la balise title 

        title = soup.title

        texte = title.get_text()
        texte = texte.replace('\xa0', ' ').strip()
        liste_texte = [x.strip() for x in texte.split(">")]

        date = liste_texte[0]

        return date

    except Exception as e:

        print("erreur  : ", e)
        print(fichier)
        return None
    
def recuperation_auteur0(fichier):
    # obtenir le code html de la page
    with open(fichier, "r", encoding = "UTF8") as f :
        html = f.read()

    # cree un objet beautifulSoup en transmettant le code html à la fonction BeautifulSoup()
    soup = BeautifulSoup(html, 'html.parser' )
    type(soup)

    # récuperer l'element dans la meta
    all_meta = soup.find_all("meta")
    auteur = all_meta[6].get("content")
    
    return auteur

def recuperation_auteur(fichier):
    # obtenir le code html de la page
    with open(fichier, "r", encoding = "UTF8") as f :
        html = f.read()

    # cree un objet beautifulSoup en transmettant le code html à la fonction BeautifulSoup()
    soup = BeautifulSoup(html, 'html.parser' )
    type(soup)

    # récuperer l'element 

    racine = soup.body.div.table
    all_span = racine.find_all("span")
    ligne = 0
    # parcourir le tableau all span jusqu'a trouver index du texte rédacteur : l'auteur est situé à l'indice suivant 
    for index, span in enumerate(all_span):
        if span.get_text().strip() == "Rédacteur :" or span.get_text().strip() == "Rédacteurs :" :
            ligne = index + 1
            break
    if ligne != 0:
        span = all_span[ligne]
        texte = span.get_text()
        traitement = [x.strip() for x in texte.split("-")]
        # print ("texte : ", texte)
        # print(traitement)
        # ça ne fonctionne plus si on change de redacteur, du coup faut essayer de revoir 
        if len(traitement)>2 :
            resultat = traitement[1] + "-" + traitement[2]
            return resultat
        else :
            print ("traitement < 2")
            return fichier
    print ("pas de ligne")
    return fichier

def recuperation_texte(fichier):
    try :
        # obtenir le code html de la page
        with open(fichier, "r", encoding = "UTF8") as f :
            html = f.read()

        # cree un objet beautifulSoup en transmettant le code html à la fonction BeautifulSoup()
        soup = BeautifulSoup(html, 'html.parser' )
        type(soup)

        # récuperer l'element 
        texte = ""
        racine = soup.body.div.table
        all_tr_niveau1 = racine.find_all("tr")
        tr_niveau1 = all_tr_niveau1[6]
        all_tr_niveau2 = tr_niveau1.td.table.find_all("tr")
        tr_niveau2 = all_tr_niveau2[2].td
        all_span = tr_niveau2.find_all("span")
        # recuperer tout les span. l'element qui caractérise le fait qu'un span soit un texte  est sa class qui est = style95
        for span in all_span:

            if span.get("class") == ["style95"]:
                texte = span.get_text() + " " + texte

        return texte
    
    except Exception as e:
        print("fichier", fichier , "erreur: ", e)
        return None

def recuperation_rubrique(fichier):
    # obtenir le code html de la page
    with open(fichier, "r", encoding = "UTF8") as f :
        html = f.read()

    # cree un objet beautifulSoup en transmettant le code html à la fonction BeautifulSoup()
    soup = BeautifulSoup(html, 'html.parser' )
    type(soup)

    # récuperer l'element 

    racine = soup.body.div.table
    all_span = racine.find_all("span")
    span = all_span[45]
    resultat = span.get_text()
    return resultat


def recuperation_information_contact(fichier):
    # obtenir le code html de la page
    with open(fichier, "r", encoding = "UTF8") as f :
        html = f.read()

    # cree un objet beautifulSoup en transmettant le code html à la fonction BeautifulSoup()
    soup = BeautifulSoup(html, 'html.parser' )
    type(soup)

    # récuperer l'element 

    racine = soup.body.div.table
    all_span = racine.find_all("span")
    ligne = 0
    # parcourir le tableau all span jusqu'a trouver index du texte rédacteur : l'auteur est situé à l'indice suivant 
    for index, span in enumerate(all_span):
        if span.get_text().strip() == "Pour en savoir plus, contacts :" or span.get_text().strip() == "Rédacteurs :Pour en savoir plus, contact :" :
            ligne = index + 1
            break
    if ligne != 0:
        span = all_span[ligne]
        resultat = span.get_text()
        return resultat
    print ("pas de ligne")
    return fichier

def recuperation_images(fichier):

    try :
        # obtenir le code html de la page
        with open(fichier, "r", encoding = "UTF8") as f :
            html = f.read()

        # cree un objet beautifulSoup en transmettant le code html à la fonction BeautifulSoup()
        soup = BeautifulSoup(html, 'html.parser' )
        type(soup)

        # récuperer l'element 
        images = {}
        racine = soup.body.div.table
        all_tr_niveau1 = racine.find_all("tr")
        tr_niveau1 = all_tr_niveau1[6]
        all_tr_niveau2 = tr_niveau1.td.table.find_all("tr")
        tr_niveau2 = all_tr_niveau2[2].td
        all_div = tr_niveau2.find_all("div")
        # recuperer les images, les mettres dans un dictionnaire. chaque dictionnaire pocede une un url et une description
        for index, div in enumerate(all_div):
            images[f"image_{index}"] = {"url" : div.img.get("src"), "description": div.span.get_text()}

        return images
    except Exception as e:
        print("fichier", fichier , "erreur: ", e)
        return None
        


# Test

In [ ]:
fichier = r"BULLETINS\67068.htm"

# obtenir le code html de la page
with open(fichier, "r", encoding = "UTF8") as f :
    html = f.read()

# cree un objet beautifulSoup en transmettant le code html à la fonction BeautifulSoup()
soup = BeautifulSoup(html, 'html.parser' )
type(soup)

# récuperer l'element 
texte = ""
racine = soup.body.div.table
all_tr_niveau1 = racine.find_all("tr")
tr_niveau1 = all_tr_niveau1[6]
all_tr_niveau2 = tr_niveau1.td.table.find_all("tr")
tr_niveau2 = all_tr_niveau2[2].td
all_span = tr_niveau2.find_all("span")
# recuperer tout les span. l'element qui caractérise le fait qu'un span soit un texte  est sa class qui est = style95
for span in all_span:

    if span.get("class") == ["style95"]:
        texte = span.get_text() + " " + texte

print(texte)

Le 27 avril dernier, le CNRS décernait pour la première fois la Médaille de l'Innovation, dont Valérie Pécresse, ministre de l'Enseignement Supérieur et de la Recherche est à l'origine de la création. Le CNRS souhaite ainsi honorer des chercheurs et ingénieurs travaillant dans des établissements publics de recherche, des universités, des grandes écoles mais aussi des industriels qui développent des innovations marquantes. Pour cette première édition, cette nouvelle distinction a été attribuée à trois chercheurs réputés : l'économiste Esther Duflo, le roboticien François Pierrot et le physicien Mathias Fink. Directeur de l'Institut Langevin "Ondes et Images", créé en janvier 2009 au sein de l'Ecole de Physique et de Chimie Industrielles de la Ville de Paris (ESPCI), Mathias Fink est un remarquable exemple de ces chercheurs qui innovent. Les travaux de son équipe ont en effet abouti à la création de quatre start-ups, qui totalisent 230 personnes, dans des domaines aussi variés que la méd

In [ ]:


def recuperation_auteur(fichier):
    # obtenir le code html de la page
    with open(fichier, "r", encoding = "UTF8") as f :
        html = f.read()

    # cree un objet beautifulSoup en transmettant le code html à la fonction BeautifulSoup()
    soup = BeautifulSoup(html, 'html.parser' )
    type(soup)

    # récuperer l'element 

    racine = soup.body.div.table
    all_span = racine.find_all("span")
    ligne = 0
    # parcourir le tableau all span jusqu'a trouver index du texte rédacteur : l'auteur est situé à l'indice suivant 
    for index, span in enumerate(all_span):
        if span.get_text().strip() == "Pour en savoir plus, contacts :" or span.get_text().strip() == "Rédacteurs :Pour en savoir plus, contact :" :
            ligne = index + 1
            break
    if ligne != 0:
        span = all_span[ligne]
        texte = span.get_text()
        traitement = [x.strip() for x in texte.split("-")]
        # print ("texte : ", texte)
        # print(traitement)
        # ça ne fonctionne plus si on change de redacteur, du coup faut essayer de revoir 
        if len(traitement)>2 :
            resultat = traitement[1] + "-" + traitement[2]
            return resultat
        else :
            print ("traitement < 2")
            return fichier
    print ("pas de ligne")
    return fichier

fichier = r"BULLETINS\70428.htm"
print(recuperation_auteur(fichier))


Jean-François Desessard


In [61]:


def recuperation_information_contact_test(fichier):
    # obtenir le code html de la page
    with open(fichier, "r", encoding = "UTF8") as f :
        html = f.read()

    # cree un objet beautifulSoup en transmettant le code html à la fonction BeautifulSoup()
    soup = BeautifulSoup(html, 'html.parser' )
    type(soup)

    # récuperer l'element 

    racine = soup.body.div.table
    all_span = racine.find_all("span")
    print("texte_98: ",all_span[98].get_text().strip())
    for index, span in enumerate(all_span):
        print(index, "-->", span.get_text())

    return all_span

fichier = r"BULLETINS\67068.htm"
print(recuperation_information_contact_test(fichier))


texte_98:  Rédacteur :
0 --> Coordonnées
1 -->  
2 --> >>
3 --> 
4 --> 
5 --> 
6 --> 
7 --> Toute l'actualité :
8 --> 
9 --> 
10 --> France
11 -->  >>
12 --> 
13 --> 
14 --> Monde
15 -->  >>
16 --> 
17 --> 
18 --> Tous les rapports :
19 --> 
20 --> 
21 --> France
22 -->  >>
23 --> 
24 --> 
25 --> Monde
26 -->  >>
27 --> 
28 --> 
29 --> 
30 --> 
31 --> 
32 --> 
33 --> Tous les flux rss
34 -->  >>
35 --> 
36 --> 
37 --> 
38 --> 
39 --> 
40 --> BE France 258  
41 --> >>
42 -->   21/06/2011
43 --> >> Sommaire
44 --> 
45 --> Focus
46 --> Physique : Mathias Fink, un bel exemple de chercheur qui innove
47 --> http://www.bulletins-electroniques.com/actualites/67068.htm
48 --> 
49 --> Le 27 avril dernier, le CNRS décernait pour la première fois la Médaille de l'Innovation, dont Valérie Pécresse, ministre de l'Enseignement Supérieur et de la Recherche est à l'origine de la création. Le CNRS souhaite ainsi honorer des chercheurs et ingénieurs travaillant dans des établissements publics de recherc

# validation des fonctions 

In [74]:
dossier = "BULLETINS"
for page in os.listdir(dossier):
    if page != "IMAGESWEB" :
        fichier = dossier + f"/{page}"
        print(recuperation_images(fichier))

{}
{}
{}
{}
{}
{}
{}
{}
{}
{}
{}
{}
{}
{}
{}
{}
{}
{}
{'image_0': {'url': 'http://www4.utc.fr/~lo17/IMAGESWEB/67794_01.jpg', 'description': "L'ouvrage de Marion Guillou et Gérard Matheron"}}
{'image_0': {'url': 'http://www4.utc.fr/~lo17/IMAGESWEB/67795_01.jpg', 'description': "Gilles Trystram, Directeur Général d'AgroParisTech"}}
{}
{}
{}
{}
{'image_0': {'url': 'http://www4.utc.fr/~lo17/IMAGESWEB/67800_01.jpg', 'description': "Quelques effets de la Galvestine-1 aux niveaux de l'enzyme-cible (la MGDG synthase, MGD), du développement subcellulaire (genèse des chloroplastes), du développement des feuilles ou de la croissance du tube pollinique"}}
{}
{}
{}
{}
{'image_0': {'url': 'http://www4.utc.fr/~lo17/IMAGESWEB/67937_01.jpg', 'description': 'Jules Hoffmann, Prix Nobel de Médecine et de Physiologie 2011'}, 'image_1': {'url': 'http://www4.utc.fr/~lo17/IMAGESWEB/67937_02.jpg', 'description': 'Jules Hoffmann : "Au départ, nous voulions comprendre comment l\'insecte se défend et bloquer ses 

In [65]:
racine = ET.Element("etudiants")
etudiant = ET.SubElement(racine, "etudiant")
nom = ET.SubElement(etudiant, "nom")
nom.text = "Dupont"

tree = ET.ElementTree(racine)
tree.write("output.xml")